# Preparação ambiente

In [ ]:
# Criação dos diretórios
from pathlib import Path

DATASET_DIR = Path("dataset-ground-truth")
SUBDIRS = [
    "audios",
    "lists",
    "rttms/train",
    "rttms/dev",
    "rttms/test",
    "uems/train",
    "uems/dev",
    "uems/test",
    "transcribe_jsons",
]
LIST_FILES = [
    "lists/train.txt",
    "lists/dev.txt",
    "lists/test.txt",
]

def ensure_dir(p: Path):
    if p.exists():
        print(f"[ok]  dir já existia: {p}")
        return False
    p.mkdir(parents=True, exist_ok=True)
    print(f"[new] dir criada:     {p}")
    return True

def ensure_file(p: Path):
    if p.exists():
        print(f"[ok]  arquivo já existia: {p}")
        return False
    p.parent.mkdir(parents=True, exist_ok=True)
    p.touch()
    print(f"[new] arquivo criado:     {p}")
    return True

def main():
    print("== Preparando estrutura do dataset ==")
    ensure_dir(DATASET_DIR)

    for sub in SUBDIRS:
        ensure_dir(DATASET_DIR / sub)

    for rel in LIST_FILES:
        ensure_file(DATASET_DIR / rel)

    print("== Pronto. Adicione seus .wav em: dataset/audios ==")

# execute
main()

# Data Preparation (AWS Transcribe)

In [ ]:
from pathlib import Path
import aioboto3
import os
import uuid
import soundfile as sf
import json
import time
import random
from botocore.exceptions import ClientError
import re
import asyncio
import boto3

# ─── CONFIGURE AQUI ─────────────────────────────────────────────────────────
# Preencha as variáveis abaixo antes de executar qualquer célula.
# Você também pode defini-las como variáveis de ambiente antes de abrir o notebook:
#   export AWS_PROFILE=meu-profile
#   export AWS_REGION=us-east-1
#   export TRANSCRIBE_BUCKET=meu-bucket
AWS_PROFILE              = os.environ.get("AWS_PROFILE", "")            # profile AWS; vazio = default da sessão
AWS_REGION               = os.environ.get("AWS_REGION", "us-east-1")   # região AWS do seu bucket e Amazon Transcribe
TRANSCRIBE_LANGUAGE_CODE = os.environ.get("TRANSCRIBE_LANG", "pt-BR")  # código BCP-47 do idioma dos áudios

bucket_name         = os.environ.get("TRANSCRIBE_BUCKET", "your-bucket-name")  # bucket S3 principal
bucket_path_input   = "audios/awstranscribe/audios"                            # prefixo S3 para upload dos áudios
bucket_path_output  = "audios/awstranscribe/transcriptions"                    # prefixo S3 para os JSONs
# ─────────────────────────────────────────────────────────────────────────────

boto3.setup_default_session(profile_name=AWS_PROFILE, region_name=AWS_REGION)

DATASET_DIR = Path("dataset")
AUDIO_DIR   = DATASET_DIR / "audios"
LISTS_DIR   = DATASET_DIR / "lists"
RTTM_DIR    = DATASET_DIR / "rttms"
UEM_DIR     = DATASET_DIR / "uems"
JSON_DIR    = DATASET_DIR / "transcribe_jsons"

MAX_CONCURRENT = 50            # quantos áudios você quer processar em paralelo
MAX_POLL_CONCURRENCY = 12       # limite de chamadas simultâneas a GetTranscriptionJob
POLL_SECS = 12.0                # intervalo base entre polls (segundos)
TIMEOUT_SECS = 3600             # timeout por job (1h)
BACKOFF_BASE = 1.0              # backoff inicial (seg)
BACKOFF_MAX  = 30.0             # backoff máximo (seg)

poll_sem = asyncio.Semaphore(MAX_POLL_CONCURRENCY)

In [ ]:
def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)

def safe_job_id(name: str) -> str:
    base = re.sub(r'[^0-9A-Za-z._-]+', '-', name).strip('-')
    if not base:
        base = uuid.uuid4().hex
    return f"{base[:180]}-{uuid.uuid4().hex[:8]}"

def aws_to_rttm(transcribe_json: dict, uri: str, out_dir: Path):
    ensure_dir(out_dir)
    out_path = out_dir / f"{uri}.rttm"

    segments = transcribe_json.get("results", {}).get("speaker_labels", {}).get("segments")
    channel_alternatives = transcribe_json.get("results", {}).get("channel_labels", {}).get("channels", [])

    lines = []

    if segments:
        for seg in segments:
            start = float(seg["start_time"])
            end = float(seg["end_time"])
            dur = end - start
            spk = seg["speaker_label"].upper()
            lines.append(f"SPEAKER {uri} 1 {start:.3f} {dur:.3f} <NA> <NA> {spk} <NA>")
    elif channel_alternatives:
        for ch in channel_alternatives:
            ch_id = ch["channel_label"]
            spk = ch_id.replace("ch_", "SPK_")
            words_alts = ch.get("alternatives", [])
            if not words_alts:
                continue
            items = words_alts[0].get("items", [])
            if not items:
                continue
            GAP = 0.75
            cur_start = None
            cur_end = None
            for w in items:
                if "start_time" not in w or "end_time" not in w:
                    continue
                w_start = float(w["start_time"]); w_end = float(w["end_time"])
                if cur_start is None:
                    cur_start, cur_end = w_start, w_end
                else:
                    if w_start - cur_end <= GAP:
                        cur_end = w_end
                    else:
                        dur = cur_end - cur_start
                        lines.append(f"SPEAKER {uri} 1 {cur_start:.3f} {dur:.3f} <NA> <NA> {spk} <NA>")
                        cur_start, cur_end = w_start, w_end
            if cur_start is not None:
                dur = cur_end - cur_start
                lines.append(f"SPEAKER {uri} 1 {cur_start:.3f} {dur:.3f} <NA> <NA> {spk} <NA>")
    else:
        raise ValueError("JSON do Transcribe não tem speaker_labels nem channel_labels.")

    with open(out_path, "w", encoding="utf-8") as out:
        out.write("\n".join(lines) + ("\n" if lines else ""))

def create_uem(uri: str, wav_path: Path, out_dir: Path):
    ensure_dir(out_dir)
    dur = sf.info(str(wav_path)).duration
    with open(out_dir / f"{uri}.uem", "w", encoding="utf-8") as f:
        f.write(f"{uri} 1 0.000 {dur:.3f}\n")

# =========================
# Funções assíncronas
# =========================
async def send_audio(session, audio_path: Path, audio_name: str):
    job_name = safe_job_id(audio_name)
    bucket_key_input = f"{bucket_path_input}/{audio_name}.wav"
    async with session.client("s3") as s3_client:
        await s3_client.upload_file(str(audio_path), bucket_name, bucket_key_input)

    s3_uri = f"s3://{bucket_name}/{bucket_key_input}"
    bucket_key_output = f"{bucket_path_output}/{job_name}.json"

    async with session.client("transcribe") as transcribe_client:
        await transcribe_client.start_transcription_job(
            TranscriptionJobName=job_name,
            LanguageCode=TRANSCRIBE_LANGUAGE_CODE,
            MediaFormat="wav",
            Media={"MediaFileUri": s3_uri},
            OutputBucketName=bucket_name,
            OutputKey=bucket_key_output,
            Settings={"ShowSpeakerLabels": True, "MaxSpeakerLabels": 2}
        )
    return job_name

async def safe_get_job(session, job_name: str):
    """
    GetTranscriptionJob com limite de concorrência + backoff exponencial com jitter
    para lidar com ThrottlingException/TooManyRequests.
    """
    delay = BACKOFF_BASE
    while True:
        try:
            async with poll_sem:
                async with session.client("transcribe") as transcribe_client:
                    return await transcribe_client.get_transcription_job(
                        TranscriptionJobName=job_name
                    )
        except ClientError as e:
            code = getattr(e, "response", {}).get("Error", {}).get("Code")
            if code in ("ThrottlingException", "TooManyRequestsException", "LimitExceededException"):
                # backoff exponencial com jitter
                sleep_for = min(BACKOFF_MAX, delay * (1.5 + random.random()))
                await asyncio.sleep(sleep_for)
                delay = min(BACKOFF_MAX, delay * 2.0)
                continue
            raise  # outros erros: propaga

async def wait_for_job_async(session, job_name: str,
                             poll_secs: float = POLL_SECS,
                             timeout_secs: int = TIMEOUT_SECS) -> str:
    start = asyncio.get_event_loop().time()
    while True:
        job = await safe_get_job(session, job_name)
        status = job["TranscriptionJob"]["TranscriptionJobStatus"]

        if status in ("COMPLETED", "FAILED"):
            return status

        if (asyncio.get_event_loop().time() - start) > timeout_secs:
            raise TimeoutError(f"Job {job_name} não terminou em {timeout_secs}s")

        # jitter leve para não bater todo mundo ao mesmo tempo
        await asyncio.sleep(poll_secs + random.uniform(0, poll_secs * 0.25))

async def wait_and_process(session, job_name: str, audio_name: str,
                           split_rttm_dir: Path, split_uem_dir: Path, audio_path: Path):
    # cria UEM já de cara (sempre útil)
    create_uem(audio_name, audio_path, split_uem_dir)

    # espera terminar com polling controlado
    status = await wait_for_job_async(session, job_name, poll_secs=POLL_SECS, timeout_secs=TIMEOUT_SECS)
    if status == "FAILED":
        print(f"❌ Job falhou: {job_name}")
        return

    # baixa JSON e valida
    async with session.client("s3") as s3_client:
        ensure_dir(JSON_DIR)
        key = f"{bucket_path_output}/{job_name}.json"
        local_file = JSON_DIR / f"{job_name}.json"
        await s3_client.download_file(bucket_name, key, str(local_file))
        with open(local_file, "r", encoding="utf-8") as f:
            tj = json.load(f)

    if not tj or not isinstance(tj, dict) or not tj.get("results"):
        print(f"⚠️ JSON inválido/sem 'results' para {audio_name}")
        return

    aws_to_rttm(tj, audio_name, split_rttm_dir)
    print(f"✅ {audio_name} concluído")

async def process_split(files, split_name):
    split_rttm_dir = RTTM_DIR / split_name
    split_uem_dir = UEM_DIR / split_name
    ensure_dir(split_rttm_dir)
    ensure_dir(split_uem_dir)

    sem = asyncio.Semaphore(MAX_CONCURRENT)
    session = aioboto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)

    async def handle(audio_path):
        async with sem:
            audio_name = audio_path.stem
            try:
                job_name = await send_audio(session, audio_path, audio_name)
                await wait_and_process(session, job_name, audio_name, split_rttm_dir, split_uem_dir, audio_path)
            except Exception as e:
                print(f"⚠️ Erro em {split_name}/{audio_name}: {e}")

    await asyncio.gather(*(handle(p) for p in files))

In [ ]:
import pandas as pd

# Planilha de anotações com as segmentações ground-truth.
# Colunas obrigatórias:
#   audios          — nome do arquivo de áudio (com ou sem .wav)
#   audio_classes   — classes/categorias do áudio (pode ser NaN)
#   label           — segmentos em JSON: [{start, end, speaker|channel, labels}]
#   label_events    — eventos opcionais (pode ser NaN)
ANNOTATIONS_FILE = os.environ.get("ANNOTATIONS_FILE", "annotations.xlsx")

df = pd.read_excel(ANNOTATIONS_FILE)
df = (
    df[['audios', 'audio_classes', 'label', 'label_events']]
    .dropna(subset=['label'])
)
df

In [ ]:
valid_audio_names = set(df['audios'].tolist())

# filtra apenas os arquivos que estão no DF
wav_files = [
    p for p in AUDIO_DIR.rglob("*.wav")
    if p.is_file() and p.name in valid_audio_names
]

if not wav_files:
    raise SystemExit("Nenhum .wav encontrado no DF e em 'dataset/audios/'")

print(f"Total de .wav válidos encontrados: {len(wav_files)}")

In [ ]:
random.seed(42)
random.shuffle(wav_files)

n = len(wav_files)
cut = max(1, int(n * 0.8)) if n > 1 else 1
train_files = wav_files[:cut]
test_files  = wav_files[cut:]

dev_cut = len(test_files) // 2
dev_files = test_files[:dev_cut]
test_files = test_files[dev_cut:]

ensure_dir(LISTS_DIR)
with open(LISTS_DIR / "train.txt", "w", encoding="utf-8") as ftrain:
    ftrain.write("\n".join([p.stem for p in train_files]) + "\n")
with open(LISTS_DIR / "dev.txt", "w", encoding="utf-8") as fdev:
    fdev.write("\n".join([p.stem for p in dev_files]) + "\n")
with open(LISTS_DIR / "test.txt", "w", encoding="utf-8") as ftest:
    ftest.write("\n".join([p.stem for p in test_files]) + "\n")

print(f"Total: {n} | train: {len(train_files)} | validation: {len(dev_files)} | test: {len(test_files)}")

# Aqui você pode rodar só uma célula no notebook:
await process_split(train_files, "train")
await process_split(dev_files, "dev")
await process_split(test_files, "test")

# Data preparation (Ground-truth)

In [ ]:
from pathlib import Path
import pandas as pd
import json, ast, math, random
import soundfile as sf

DATASET_DIR = Path("dataset-ground-truth")
AUDIO_DIR   = DATASET_DIR / "audios"
LISTS_DIR   = DATASET_DIR / "lists"
RTTM_DIR    = DATASET_DIR / "rttms"
UEM_DIR     = DATASET_DIR / "uems"

In [ ]:
def ensure_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)

# ---------- helpers p/ parsing ----------
def _to_python(obj):
    """Converte str JSON/str literal -> objeto Python; mantém list/dict; trata NaN."""
    if obj is None or (isinstance(obj, float) and math.isnan(obj)):
        return None
    if isinstance(obj, (list, dict)):
        return obj
    if isinstance(obj, str):
        s = obj.strip()
        if not s:
            return None
        try:
            return json.loads(s)      # tenta JSON
        except Exception:
            pass
        try:
            return ast.literal_eval(s)  # tenta aspas simples
        except Exception:
            return None
    return None

def parse_segments_from_label(label_value):
    """
    Espera lista de segmentos com: start, end, (speaker|channel).
    Retorna [{'start': float, 'end': float, 'speaker': 'SPK_X'}, ...] ordenados por start.
    """
    parsed = _to_python(label_value)
    if not parsed:
        return []

    # às vezes vem {"segments": [...]} — aceita também
    if isinstance(parsed, dict) and "segments" in parsed:
        parsed = parsed["segments"]

    out = []
    if isinstance(parsed, list):
        for seg in parsed:
            if not isinstance(seg, dict):
                continue
            if "start" not in seg or "end" not in seg:
                continue
            try:
                start = float(seg["start"]); end = float(seg["end"])
            except Exception:
                continue
            if end <= start:
                continue

            if "speaker" in seg and seg["speaker"] not in (None, ""):
                spk = str(seg["speaker"]).upper()
            elif "channel" in seg:
                try:
                    if seg['labels'][0] == 'Interlocutor 1':
                        spk = f"SPK_0"
                    if seg['labels'][0] == 'Interlocutor 2':
                        spk = f"SPK_1"
                except Exception:
                    spk = "SPK_0"
            else:
                spk = "SPK_0"

            out.append({"start": start, "end": end, "speaker": spk})

    return sorted(out, key=lambda s: s["start"])

def write_rttm(uri: str, segments, out_dir: Path):
    ensure_dir(out_dir)
    lines = []
    for seg in segments:
        start = float(seg["start"])
        dur = float(seg["end"]) - start
        spk = seg["speaker"]
        if dur <= 0:
            continue
        # SPEAKER <uri> <chnl> <tbeg> <tdur> <ortho> <stype> <name> <conf>
        lines.append(f"SPEAKER {uri} 1 {start:.3f} {dur:.3f} <NA> <NA> {spk} <NA>")
    (out_dir / f"{uri}.rttm").write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")

def write_uem(uri: str, wav_path: Path, out_dir: Path):
    ensure_dir(out_dir)
    dur = sf.info(str(wav_path)).duration
    (out_dir / f"{uri}.uem").write_text(f"{uri} 1 0.000 {dur:.3f}\n", encoding="utf-8")

# ---------- pipeline principal (sem AWS) ----------
def generate_from_df(df: pd.DataFrame, audio_col="audios", label_col="label"):
    # mapeia nome-base -> label
    def base(name: str) -> str:
        p = Path(str(name))
        return p.stem  # remove extensão caso exista

    df_local = df.dropna(subset=[label_col]).copy()
    name_to_label = {base(row[audio_col]): row[label_col] for _, row in df_local.iterrows()}

    # Filta apenas os nomes válidos
    valid_names = set(Path(a).stem for a in df[audio_col])

    # lista os .wav e cria splits como antes
    wav_files = [p for p in AUDIO_DIR.rglob("*.wav") if p.stem in valid_names]
    if not wav_files:
        raise SystemExit("Nenhum .wav encontrado em 'dataset/audios/'")

    random.seed(42)
    random.shuffle(wav_files)

    n = len(wav_files)
    cut = max(1, int(n * 0.8)) if n > 1 else 1
    train_files = wav_files[:cut]
    test_files  = wav_files[cut:]
    dev_cut = len(test_files) // 2
    dev_files = test_files[:dev_cut]
    test_files = test_files[dev_cut:]

    ensure_dir(LISTS_DIR)
    (LISTS_DIR / "train.txt").write_text("\n".join([p.stem for p in train_files]) + "\n", encoding="utf-8")
    (LISTS_DIR / "dev.txt").write_text("\n".join([p.stem for p in dev_files]) + "\n", encoding="utf-8")
    (LISTS_DIR / "test.txt").write_text("\n".join([p.stem for p in test_files]) + "\n", encoding="utf-8")

    print(f"Total: {n} | train: {len(train_files)} | validation: {len(dev_files)} | test: {len(test_files)}")

    # processa rttm/uem por split
    for split_name, files in (("train", train_files), ("dev", dev_files), ("test", test_files)):
        split_rttm_dir = RTTM_DIR / split_name
        split_uem_dir  = UEM_DIR / split_name
        ensure_dir(split_rttm_dir); ensure_dir(split_uem_dir)

        for audio_path in files:
            uri = audio_path.stem
            label_value = name_to_label.get(uri)
            if not label_value:
                print(f"⚠️  pulando {split_name}/{uri}: sem 'label' no DataFrame")
                continue

            try:
                segments = parse_segments_from_label(label_value)
                write_rttm(uri, segments, split_rttm_dir)
                write_uem(uri, audio_path, split_uem_dir)
                print(f"✅ {split_name}/{uri} gerado (RTTM+UEM)")
            except Exception as e:
                print(f"⚠️  erro em {split_name}/{uri}: {e}")

In [ ]:
import pandas as pd

# Planilha de anotações com as segmentações ground-truth.
# Colunas obrigatórias:
#   audios          — nome do arquivo de áudio (com ou sem .wav)
#   audio_classes   — classes/categorias do áudio (pode ser NaN)
#   label           — segmentos em JSON: [{start, end, speaker|channel, labels}]
#   label_events    — eventos opcionais (pode ser NaN)
ANNOTATIONS_FILE = os.environ.get("ANNOTATIONS_FILE", "annotations.xlsx")

df = pd.read_excel(ANNOTATIONS_FILE)
df = (
    df[['audios', 'audio_classes', 'label', 'label_events']]
    .dropna(subset=['label'])
)
df

In [ ]:
# Lista de arquivos no disco
audio_files = [p.stem for p in AUDIO_DIR.glob("*.wav")]
audio_files_set = set(audio_files)

# Lista de nomes do DF
df_audio_names = {Path(name).stem for name in df["audios"].astype(str).str.strip()}

# Contagens
total_no_disco = len(audio_files_set)
total_no_df = len(df_audio_names)

# Interseção e diferenças
present_in_both = audio_files_set & df_audio_names
missing_in_disk = df_audio_names - audio_files_set
missing_in_df = audio_files_set - df_audio_names

print(f"Áudios no disco: {total_no_disco}")
print(f"Nomes únicos no DF: {total_no_df}")
print(f"Presentes nos dois: {len(present_in_both)}")
print(f"Faltando no disco (presentes no DF mas não no diretório): {len(missing_in_disk)}")
print(f"Faltando no DF (presentes no disco mas não na planilha): {len(missing_in_df)}")

if missing_in_disk:
    print("\n⚠️ No DF mas não no disco:")
    for name in sorted(missing_in_disk):
        print(f"  {name}")

if missing_in_df:
    print("\n⚠️ No disco mas não no DF:")
    for name in sorted(missing_in_df):
        print(f"  {name}")

In [ ]:
generate_from_df(df)

# Validate Database

Remove os áudios problemáticos, com:
 - uem fora do intervalo do áudio
 - rttms ultrapassando os limites do uem

In [ ]:
def validar_dataset_limpar(base_path="dataset-ground-truth"):
    problems = []
    to_remove = {"train": set(), "dev": set(), "test": set()}
    
    for split in ["train", "dev", "test"]:
        list_file = os.path.join(base_path, "lists", f"{split}.txt")
        
        if not os.path.exists(list_file):
            problems.append(f"[{split}] Lista {list_file} não encontrada.")
            continue
        
        with open(list_file, "r") as f:
            uris = [line.strip() for line in f if line.strip()]
        
        for uri in uris:
            audio_path = os.path.join(base_path, "audios", f"{uri}.wav")
            rttm_path = os.path.join(base_path, "rttms", split, f"{uri}.rttm")
            uem_path = os.path.join(base_path, "uems", split, f"{uri}.uem")

            has_problem = False
            
            # Verificar existência
            for path in [audio_path, rttm_path, uem_path]:
                if not os.path.exists(path):
                    problems.append(f"[{split}] {uri} → Arquivo faltando: {path}")
                    has_problem = True
            
            # Checar duração do áudio
            duration = None
            if os.path.exists(audio_path):
                try:
                    with sf.SoundFile(audio_path) as f:
                        duration = len(f) / f.samplerate
                except Exception as e:
                    problems.append(f"[{split}] {uri} → Erro ao ler áudio: {e}")
                    has_problem = True
            
            # Checar se RTTM tem conteúdo válido
            if os.path.exists(rttm_path):
                with open(rttm_path) as f:
                    rttm_lines = [l.strip() for l in f if l.strip() and not l.startswith("#")]
                if not rttm_lines:
                    problems.append(f"[{split}] {uri} → RTTM vazio.")
                    has_problem = True
                elif duration:
                    for line in rttm_lines:
                        parts = line.split()
                        if len(parts) >= 5:
                            start_time = float(parts[3])
                            seg_duration = float(parts[4])
                            if start_time + seg_duration > duration:
                                problems.append(f"[{split}] {uri} → Segmento RTTM fora do tempo do áudio.")
                                has_problem = True
            
            # Checar se UEM tem conteúdo válido
            if os.path.exists(uem_path):
                with open(uem_path) as f:
                    uem_lines = [l.strip() for l in f if l.strip() and not l.startswith("#")]
                if not uem_lines:
                    problems.append(f"[{split}] {uri} → UEM vazio.")
                    has_problem = True
                elif duration:
                    for line in uem_lines:
                        parts = line.split()
                        if len(parts) >= 4:
                            start_time = float(parts[2])
                            end_time = float(parts[3])
                            if end_time > duration:
                                problems.append(f"[{split}] {uri} → Segmento UEM fora do tempo do áudio.")
                                has_problem = True

            # Se deu problema, marca para remover
            if has_problem:
                to_remove[split].add(uri)

    # # Remove os arquivos problemáticos e atualiza listas
    # for split, uris in to_remove.items():
    #     if not uris:
    #         continue

    #     print(f"\n🗑 Removendo {len(uris)} itens problemáticos de {split}...")

    #     list_file = os.path.join(base_path, "lists", f"{split}.txt")
    #     with open(list_file, "r") as f:
    #         all_uris = [line.strip() for line in f if line.strip()]
        
    #     # Filtra fora os problemáticos
    #     new_uris = [u for u in all_uris if u not in uris]
    #     with open(list_file, "w") as f:
    #         f.write("\n".join(new_uris) + "\n")

    #     # Apaga RTTM e UEM
    #     for uri in uris:
    #         for ext, folder in [("rttm", "rttms"), ("uem", "uems")]:
    #             path = os.path.join(base_path, folder, split, f"{uri}.{ext}")
    #             if os.path.exists(path):
    #                 os.remove(path)

    if problems:
        print("\n⚠️ Problemas encontrados e corrigidos:")
        for p in problems:
            print(" -", p)
    else:
        print("✅ Dataset validado sem problemas.")

# Rodar
validar_dataset_limpar()

# Dataset publication to huggingface datasets

In [ ]:
from datasets import Dataset, DatasetDict, Audio
from pathlib import Path
import json

AUDIO_DIR = Path("dataset-ground-truth/audios")
RTTM_DIR  = Path("dataset-ground-truth/rttms")
LISTS_DIR = Path("dataset-ground-truth/lists")

def parse_rttm(rttm_file):
    segments = []
    with open(rttm_file, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 8: 
                continue
            start = float(parts[3])
            dur   = float(parts[4])
            end   = start + dur
            speaker = parts[7]
            segments.append({"start": start, "end": end, "speaker": speaker})
    return segments

def build_split(split):
    uris = open(LISTS_DIR / f"{split}.txt").read().splitlines()
    examples = []
    for uri in uris:
        wav_path = AUDIO_DIR / f"{uri}.wav"
        rttm_path = RTTM_DIR / split / f"{uri}.rttm"
        if not wav_path.exists() or not rttm_path.exists():
            continue

        segments = parse_rttm(rttm_path)

        timestamps_start = [s["start"] for s in segments]
        timestamps_end   = [s["end"] for s in segments]
        speakers         = [s["speaker"] for s in segments]

        examples.append({
            "uri": uri,
            "audio": str(wav_path),
            "timestamps_start": timestamps_start,
            "timestamps_end": timestamps_end,
            "speakers": speakers,
        })
    return Dataset.from_list(examples)

train_ds = build_split("train")
dev_ds   = build_split("dev")
test_ds  = build_split("test")

# converte coluna audio pro tipo correto
train_ds = train_ds.cast_column("audio", Audio(sampling_rate=16000))
dev_ds   = dev_ds.cast_column("audio", Audio(sampling_rate=16000))
test_ds  = test_ds.cast_column("audio", Audio(sampling_rate=16000))

dataset = DatasetDict({
    "train": train_ds,
    "validation": dev_ds,
    "test": test_ds
})

print(dataset)


In [ ]:
from datasets import DatasetDict

# suponha que você já tem train_ds, dev_ds, test_ds
dataset = DatasetDict({
    "train": train_ds,
    "validation": dev_ds,
    "test": test_ds,
})

# Substitua pelo nome do seu repositório no Hugging Face Hub
# Formato: "seu-usuario/nome-do-dataset"  |  Exemplo: "myorg/diarization-dataset"
HF_DATASET_REPO = os.environ.get("HF_DATASET_REPO", "seu-usuario/nome-do-dataset")
dataset.push_to_hub(HF_DATASET_REPO)

In [ ]:
from datasets import load_dataset
ds = load_dataset(os.environ.get("HF_DATASET_REPO", "seu-usuario/nome-do-dataset"))

In [ ]:
print(ds)